# PDF scan (OpenAI Vector Store + EVIDENCE-Pass)

Dieses Notebook:
1) lädt ein PDF in einen OpenAI Vector Store (hosted)
2) holt EVIDENCE-Auszüge via `vector_stores.search` und extrahiert daraus passende Stellen für dein Kapitel
3) gibt Treffer als **Suchanker** (für Strg+F) + Kurz-Zusammenfassung + Score aus (ohne Seitenzahlen)
4) zeigt am Ende eine Kostenübersicht (Token-basiert) für `gpt-5-nano`, `gpt-5-mini`, `gpt-5.2`

Voraussetzung: `OPENAI_API_KEY` ist in der Repo-`.env` oder als Umgebungsvariable gesetzt.


In [1]:
import json
import os
import time
from pathlib import Path
from typing import Any, Dict, Optional

from dotenv import load_dotenv
from openai import OpenAI

# .env laden (Repo root)
load_dotenv('.env', override=True)

api_key = (os.getenv('OPENAI_API_KEY') or '').strip()
if not api_key:
    raise RuntimeError('OPENAI_API_KEY fehlt. Setze ihn in der .env oder als Umgebungsvariable.')

client = OpenAI(api_key=api_key)

# --- Config ---
PDF_PATH = r"C:\\path\\to\\your.pdf"  # TODO

# Optional: existierenden Vector Store / File wiederverwenden (spart erneutes Upload + Indexing)
# Du kannst die IDs auch über die Env-Variablen setzen.
VECTOR_STORE_ID = (os.getenv('OPENAI_VECTOR_STORE_ID', '').strip() or 'vs_6981328185b081919dddaad9c0b3d2d7')
PDF_FILE_ID = (os.getenv('OPENAI_PDF_FILE_ID', '').strip() or 'file-9Jj149XjyGGqfr3HHb8NvT')

# Modelle
MODEL = (os.getenv('OPENAI_PDF_SCAN_MODEL', 'gpt-5-mini') or '').strip() or 'gpt-5-mini'
PREPROCESS_MODEL = (os.getenv('OPENAI_PDF_SCAN_PREPROCESS_MODEL', 'gpt-5-nano') or '').strip() or 'gpt-5-nano'

# Modelle pro Stage (einfach ändern)
# Beispiele:
#   STAGE_MODELS['preprocess'] = 'gpt-5-nano'
#   STAGE_MODELS['pdf_search'] = 'gpt-5.2'
STAGE_MODELS = {
    'preprocess': PREPROCESS_MODEL,
    'pdf_search': MODEL,
}


def model_for_stage(stage: str, default: Optional[str] = None) -> str:
    # 1) Notebook override via STAGE_MODELS
    v = (STAGE_MODELS.get(stage) or '').strip()
    if v:
        return v

    # 2) Optional env override, e.g. OPENAI_MODEL_PREPROCESS / OPENAI_MODEL_PDF_SEARCH
    env_key = f"OPENAI_MODEL_{stage.upper()}"
    v = (os.getenv(env_key) or '').strip()
    if v:
        return v

    # 3) Fallback
    return (default or MODEL or '').strip() or 'gpt-5-mini'

# Optional: Kapitelbeschreibung per LLM komprimieren/strukturieren
ENABLE_LLM_PREPROCESS = True

# Wichtig für gpt-5*: wenn max_output_tokens gesetzt ist, kann das Modell sonst in reines "reasoning"
# laufen und keine Antwort ausgeben. Mit reasoning.effort='low' kommt zuverlässig Output.
REASONING_EFFORT = (os.getenv('OPENAI_REASONING_EFFORT', 'low') or 'low').strip()

# Retrieval/Output Tuning
FILE_SEARCH_MAX_RESULTS = 20
MAX_HITS = 8
MAX_OUTPUT_TOKENS = 2500

# Preprocess Output-Limit (hoch genug, damit JSON nicht abgeschnitten wird)
PREPROCESS_MAX_OUTPUT_TOKENS = int(os.getenv('OPENAI_PREPROCESS_MAX_OUTPUT_TOKENS', '1200') or '1200')
PREPROCESS_RETRY_MAX_OUTPUT_TOKENS = int(os.getenv('OPENAI_PREPROCESS_RETRY_MAX_OUTPUT_TOKENS', '2000') or '2000')

# --- Pricing (USD / 1M tokens) ---
# Quelle: OpenAI Pricing (Stand: 2026-02-02)
MODEL_PRICING_USD_PER_1M = {
    'gpt-5-nano': {'input': 0.05, 'cached_input': 0.005, 'output': 0.40},
    'gpt-5-mini': {'input': 0.25, 'cached_input': 0.025, 'output': 2.00},
    'gpt-5.2': {'input': 1.75, 'cached_input': 0.175, 'output': 14.00},
}

COST_EVENTS = []  # wird über record_cost_event(...) befüllt


def _usage_int(obj: Any, key: str, default: int = 0) -> int:
    if obj is None:
        return default
    if isinstance(obj, dict):
        return int(obj.get(key, default) or 0)
    return int(getattr(obj, key, default) or 0)


def extract_usage(response: Any) -> Dict[str, int]:
    usage = getattr(response, 'usage', None)
    input_tokens = _usage_int(usage, 'input_tokens', 0)
    output_tokens = _usage_int(usage, 'output_tokens', 0)

    input_details = None
    if isinstance(usage, dict):
        input_details = usage.get('input_tokens_details')
    else:
        input_details = getattr(usage, 'input_tokens_details', None)

    cached_input_tokens = _usage_int(input_details, 'cached_tokens', 0)
    return {
        'input_tokens': int(input_tokens),
        'cached_input_tokens': int(cached_input_tokens),
        'output_tokens': int(output_tokens),
    }


def estimate_cost_usd(model: str, usage: Dict[str, int]) -> Optional[float]:
    prices = MODEL_PRICING_USD_PER_1M.get(model)
    if not prices:
        return None
    input_tokens = int(usage.get('input_tokens', 0) or 0)
    cached = int(usage.get('cached_input_tokens', 0) or 0)
    output_tokens = int(usage.get('output_tokens', 0) or 0)

    billable_input = max(0, input_tokens - cached)
    cost = (
        billable_input * float(prices['input'])
        + cached * float(prices['cached_input'])
        + output_tokens * float(prices['output'])
    ) / 1_000_000.0
    return float(cost)


def estimate_costs_for_all_priced_models(usage: Dict[str, int]) -> Dict[str, Optional[float]]:
    return {m: estimate_cost_usd(m, usage) for m in MODEL_PRICING_USD_PER_1M.keys()}


def record_cost_event(stage: str, response: Any) -> None:
    usage = extract_usage(response)
    COST_EVENTS.append(
        {
            'stage': stage,
            'model': getattr(response, 'model', None),
            'usage': usage,
            'costs_usd': estimate_costs_for_all_priced_models(usage),
        }
    )


def fmt_usd(x: Optional[float]) -> str:
    if x is None:
        return 'n/a'
    return f"${x:.6f}"


def get_response_text(response: Any) -> str:
    text = getattr(response, 'output_text', None)
    if isinstance(text, str) and text.strip():
        return text

    output = getattr(response, 'output', None)
    if output:
        for out_item in output:
            if isinstance(out_item, dict):
                content = out_item.get('content') or []
            else:
                content = getattr(out_item, 'content', None) or []
            for c in content:
                if isinstance(c, dict):
                    t = c.get('text')
                    if isinstance(t, str) and t.strip():
                        return t
                    if c.get('json') is not None:
                        return json.dumps(c.get('json'))
                    if c.get('parsed') is not None:
                        return json.dumps(c.get('parsed'))
                else:
                    t = getattr(c, 'text', None)
                    if isinstance(t, str) and t.strip():
                        return t
                    j = getattr(c, 'json', None)
                    if j is not None:
                        return json.dumps(j)
                    p = getattr(c, 'parsed', None)
                    if p is not None:
                        return json.dumps(p)

    return text if isinstance(text, str) else ''


def response_error_message(response: Any) -> Optional[str]:
    err = getattr(response, 'error', None)
    if not err:
        return None
    if isinstance(err, dict):
        msg = err.get('message')
        return (msg or str(err)).strip()
    msg = getattr(err, 'message', None)
    return (msg or str(err)).strip()


def poll_response_until_output(
    client: Any,
    response: Any,
    *,
    stage: str,
    timeout_s: int = 180,
    poll_interval_s: float = 1.0,
) -> Any:
    rid = getattr(response, 'id', None)
    if not rid:
        return response

    start = time.time()
    while True:
        if response_error_message(response):
            return response
        if (get_response_text(response) or '').strip():
            return response

        status = getattr(response, 'status', None)
        if status in {'completed', 'failed', 'cancelled', 'incomplete'}:
            return response

        if (time.time() - start) >= float(timeout_s):
            return response

        time.sleep(poll_interval_s)
        try:
            response = client.responses.retrieve(rid)
        except Exception:
            return response


def build_evidence_from_vector_store_search(
    search_page: Any,
    *,
    max_hits: int = 12,
    max_chars_per_hit: int = 1800,
) -> str:
    items = getattr(search_page, 'data', None)
    if items is None:
        try:
            items = list(search_page)
        except Exception:
            items = []

    evidence_parts = []
    for i, item in enumerate(list(items)[: max(0, int(max_hits))], start=1):
        score = getattr(item, 'score', None)
        if score is None:
            score = getattr(item, 'relevance_score', None)

        parts = []
        content = getattr(item, 'content', None)
        if content:
            for c in content:
                if isinstance(c, dict):
                    t = c.get('text')
                    if isinstance(t, str) and t.strip():
                        parts.append(t)
                else:
                    t = getattr(c, 'text', None)
                    if isinstance(t, str) and t.strip():
                        parts.append(t)

        text = '\n'.join(parts).strip()
        if not text:
            t = getattr(item, 'text', None)
            if isinstance(t, str) and t.strip():
                text = t.strip()

        text = ' '.join(text.split())
        if not text:
            continue

        if len(text) > int(max_chars_per_hit):
            text = text[: max(0, int(max_chars_per_hit))]
            if ' ' in text:
                text = text.rsplit(' ', 1)[0]
            text = text.strip()

        score_str = f"{float(score):.3f}" if score is not None else 'n/a'
        evidence_parts.append(f"[EVIDENCE {i} | score={score_str}]\n{text}")

    return '\n\n'.join(evidence_parts)


def parse_json_from_response(response: Any, stage: str) -> Dict[str, Any]:
    err_msg = response_error_message(response)
    if err_msg:
        rid = getattr(response, 'id', None)
        model = getattr(response, 'model', None)
        status = getattr(response, 'status', None)
        raise RuntimeError(f"{stage}: API error (id={rid}, model={model}, status={status}): {err_msg}")

    # Prefer structured outputs if present
    output = getattr(response, 'output', None)
    if output:
        for out_item in output:
            content = out_item.get('content') if isinstance(out_item, dict) else getattr(out_item, 'content', None)
            if not content:
                continue
            for c in content:
                if isinstance(c, dict):
                    if c.get('parsed') is not None:
                        return c.get('parsed')
                    if c.get('json') is not None:
                        return c.get('json')
                else:
                    p = getattr(c, 'parsed', None)
                    if p is not None:
                        return p
                    j = getattr(c, 'json', None)
                    if j is not None:
                        return j

    raw = (get_response_text(response) or '').strip()
    if not raw:
        rid = getattr(response, 'id', None)
        model = getattr(response, 'model', None)
        status = getattr(response, 'status', None)
        inc = getattr(response, 'incomplete_details', None)
        raise RuntimeError(f"{stage}: empty model output (id={rid}, model={model}, status={status}, incomplete_details={inc}).")

    # Strip common code-fence wrappers
    if raw.startswith('```'):
        raw = raw.split('\n', 1)[1] if '\n' in raw else ''
        raw = raw.rsplit('```', 1)[0] if '```' in raw else raw
        raw = raw.strip()

    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        # Robust fallback: sometimes models emit multiple JSON values (e.g. '{}{}').
        # We parse all JSON values we can find and pick the best candidate.
        decoder = json.JSONDecoder()

        starts = [0]
        for ch in ('{', '['):
            pos = raw.find(ch)
            if pos != -1:
                starts.append(pos)

        candidates = []
        for start in sorted(set(starts)):
            idx = int(start)
            local = []
            while idx < len(raw):
                while idx < len(raw) and raw[idx].isspace():
                    idx += 1
                if idx >= len(raw):
                    break
                try:
                    obj, end = decoder.raw_decode(raw, idx)
                except json.JSONDecodeError:
                    break
                local.append(obj)
                idx = int(end)
            if local:
                candidates = local
                break

        stage_l = (stage or '').lower()
        expected_keys = None
        if 'preprocess' in stage_l:
            expected_keys = ['optimized_description', 'subpoints', 'preferred_search_terms', 'hard_exclusions']
        elif 'search' in stage_l:
            expected_keys = ['none_found', 'results']

        def has_expected_keys(obj: Any) -> bool:
            if not expected_keys:
                return False
            return isinstance(obj, dict) and all(k in obj for k in expected_keys)

        for cand in reversed(candidates):
            if has_expected_keys(cand):
                return cand

        dict_candidates = [c for c in candidates if isinstance(c, dict)]
        if dict_candidates:
            merged = {}
            for c in dict_candidates:
                merged.update(c)
            if has_expected_keys(merged):
                return merged
            return dict_candidates[-1]

        raise


In [2]:
# --- Kapitel Input ---
# Du kannst diese beiden Variablen frei ändern.
# Hinweis: TITLE und DESCRIPTION werden später zu einem Such-Prompt kombiniert.
# In die DESCRIPTION gehört nur der inhaltliche Kapitel-Text (keine Such-/Output-Anweisungen).

USE_INTERACTIVE_INPUT = False

CHAPTER_TITLE = "Technische Grundlagen: Zero Trust Architecture (ZTA) in Unternehmensnetzwerken"

<Prompt entfernt: wird zur Laufzeit aus Firebase geladen>
"""

if USE_INTERACTIVE_INPUT:
    CHAPTER_TITLE = (input('Kapitel-Titel: ') or '').strip()
    print('Beschreibung eingeben (beenden mit leerer Zeile):')
    lines = []
    while True:
        line = input()
        if not line.strip():
            break
        lines.append(line)
    CHAPTER_DESCRIPTION = "\n".join(lines).strip()

if not CHAPTER_TITLE.strip():
    raise ValueError('CHAPTER_TITLE ist leer')
if not CHAPTER_DESCRIPTION.strip():
    raise ValueError('CHAPTER_DESCRIPTION ist leer')


In [3]:
# --- Preprocessing ---
# Ziel: lange Kapitelbeschreibung -> kompakte, gut durchsuchbare "Search Spec".

def normalize_whitespace(text: str) -> str:
    text = (text or '').replace('\r\n', '\n').replace('\r', '\n')
    lines = [ln.strip() for ln in text.split('\n')]
    lines = [ln for ln in lines if ln]
    return '\n'.join(lines).strip()


OPTIMIZED_DESCRIPTION = normalize_whitespace(CHAPTER_DESCRIPTION)
SUBPOINTS = []
PREFERRED_SEARCH_TERMS = []
HARD_EXCLUSIONS = []

if ENABLE_LLM_PREPROCESS:
    try:
        preprocess_schema = {
            'type': 'object',
            'additionalProperties': False,
            'properties': {
                'optimized_description': {
                    'type': 'string',
                    'description': 'Kompakte, such-optimierte Kapitelbeschreibung (Deutsch, max ~1200 Zeichen).',
                    'maxLength': 1200,
                },
                'subpoints': {
                    'type': 'array',
                    'description': 'Unterpunkte wie (2.1) ... (falls vorhanden).',
                    'maxItems': 20,
                    'items': {
                        'type': 'object',
                        'additionalProperties': False,
                        'properties': {
                            'id': {'type': 'string', 'maxLength': 20},
                            'label': {'type': 'string', 'maxLength': 180},
                            'keywords': {
                                'type': 'array',
                                'maxItems': 12,
                                'items': {'type': 'string', 'maxLength': 60},
                            },
                        },
                        'required': ['id', 'label', 'keywords'],
                    },
                },
                'preferred_search_terms': {
                    'type': 'array',
                    'description': 'Kurze Liste von Keywords/Synonymen für Retrieval.',
                    'maxItems': 25,
                    'items': {'type': 'string', 'maxLength': 60},
                },
                'hard_exclusions': {
                    'type': 'array',
                    'description': 'Was explizit NICHT rein soll (kurz).',
                    'maxItems': 25,
                    'items': {'type': 'string', 'maxLength': 80},
                },
            },
            'required': ['optimized_description', 'subpoints', 'preferred_search_terms', 'hard_exclusions'],
        }

        system = (
<Prompt entfernt: wird zur Laufzeit aus Firebase geladen>            'Search-Spezifikation für semantische PDF-Suche erstellt.\n'
            'Regeln:\n'
            '- Schreibe auf Deutsch.\n'
            '- Maximal ~1200 Zeichen in optimized_description.\n'
            '- Extrahiere Unterpunkte (z.B. 2.1, 2.2, ...) falls im Text vorhanden; sonst leere Liste.\n'
            '- Füge sinnvolle Synonyme/Keywords hinzu, aber erfinde keine neuen Themen außerhalb des Scopes.\n'
            '- Gib NUR JSON zurück, passend zum Schema.'
        )

        user = (
            f'Kapitel-Titel: {CHAPTER_TITLE}\n\n'
            f'Rohbeschreibung:\n{normalize_whitespace(CHAPTER_DESCRIPTION)}\n'
        )

        def run_preprocess_attempt(max_out: int, stage_name: str) -> Dict[str, Any]:
            resp = client.responses.create(
                background=False,
                model=model_for_stage('preprocess', PREPROCESS_MODEL),
                reasoning={'effort': REASONING_EFFORT},
                input=[
                    {'role': 'system', 'content': [{'type': 'input_text', 'text': system}]},
                    {'role': 'user', 'content': [{'type': 'input_text', 'text': user}]},
                ],
                text={
                    'format': {
                        'type': 'json_schema',
                        'name': 'chapter_preprocess',
                        'schema': preprocess_schema,
                        'strict': True,
                    }
                },
                max_output_tokens=int(max_out),
            )
            resp = poll_response_until_output(client, resp, stage=stage_name)
            record_cost_event(stage_name, resp)
            return parse_json_from_response(resp, stage_name)

        pre_json = None
        try:
            pre_json = run_preprocess_attempt(PREPROCESS_MAX_OUTPUT_TOKENS, 'preprocess')
        except Exception as e:
            print('Warning: LLM preprocessing failed; retrying once with higher max_output_tokens:', e)
            pre_json = run_preprocess_attempt(PREPROCESS_RETRY_MAX_OUTPUT_TOKENS, 'preprocess_retry')

        OPTIMIZED_DESCRIPTION = (pre_json.get('optimized_description') or '').strip() or OPTIMIZED_DESCRIPTION
        SUBPOINTS = pre_json.get('subpoints') or []
        PREFERRED_SEARCH_TERMS = pre_json.get('preferred_search_terms') or []
        HARD_EXCLUSIONS = pre_json.get('hard_exclusions') or []
    except Exception as e:
        print('Warning: LLM preprocessing failed; continuing without it:', e)

print('--- Optimized description ---')
print(OPTIMIZED_DESCRIPTION)
print('\n--- Subpoints ---')
for sp in SUBPOINTS:
    print(f"- {sp.get('id')}: {sp.get('label')}")
print('\n--- Preferred search terms ---')
print(', '.join(PREFERRED_SEARCH_TERMS[:30]))
print('\n--- Hard exclusions ---')
print(', '.join(HARD_EXCLUSIONS[:30]))


--- Optimized description ---
<Prompt entfernt: wird zur Laufzeit aus Firebase geladen>
--- Subpoints ---

--- Preferred search terms ---


--- Hard exclusions ---



In [ ]:
# --- Upload / Indexing ---

pdf_path = Path(PDF_PATH).expanduser().resolve()

# Lokales PDF ist nur nötig, wenn du kein bestehendes OPENAI File nutzt.
if not (PDF_FILE_ID or '').strip():
    if not pdf_path.exists():
        raise FileNotFoundError(f"PDF not found: {pdf_path}")

pdf_display_name = pdf_path.name if pdf_path.exists() else (PDF_FILE_ID or 'pdf')

# 1) Vector Store erstellen (falls nicht gesetzt)
if not (VECTOR_STORE_ID or '').strip():
    vector_store = client.vector_stores.create(
        name=f"pdf-scan:{pdf_display_name}",
        # Optional: automatisch ablaufen lassen, damit keine dauerhaften Storage-Kosten entstehen
        expires_after={"anchor": "last_active_at", "days": 30},
    )
    VECTOR_STORE_ID = vector_store.id
    print('Created vector store:', VECTOR_STORE_ID)
else:
    print('Using existing vector store:', VECTOR_STORE_ID)

# 2) File bestimmen (reuse oder upload)
file_id = (PDF_FILE_ID or '').strip()
if file_id:
    print('Reusing existing file_id:', file_id)
else:
    with pdf_path.open('rb') as f:
        file_obj = client.files.create(file=f, purpose='assistants')
    file_id = file_obj.id
    print('Uploaded file_id:', file_id)

# 3) Sicherstellen, dass das File am Vector Store hängt (und fertig indiziert ist)
already_attached = False
try:
    existing = client.vector_stores.files.list(vector_store_id=VECTOR_STORE_ID, limit=100)
    for item in (getattr(existing, 'data', None) or []):
        if getattr(item, 'id', None) == file_id or getattr(item, 'file_id', None) == file_id:
            already_attached = True
            break
except Exception as e:
    print('Warning: could not list vector store files (continuing):', e)

if already_attached:
    print('File already attached to vector store.')
    try:
        vs_file = client.vector_stores.files.retrieve(vector_store_id=VECTOR_STORE_ID, file_id=file_id)
        status = getattr(vs_file, 'status', None)
        if status and status not in {'completed', 'failed'} and hasattr(client.vector_stores.files, 'poll'):
            vs_file = client.vector_stores.files.poll(vector_store_id=VECTOR_STORE_ID, file_id=file_id)
            status = getattr(vs_file, 'status', status)
        print('Vector store file status:', status)
        if status == 'failed':
            raise RuntimeError('Vector store indexing failed. Check the OpenAI dashboard / API response details.')
    except Exception as e:
        print('Warning: could not retrieve/poll vector store file status:', e)
else:
    if hasattr(client.vector_stores.files, 'create_and_poll'):
        try:
            vs_file = client.vector_stores.files.create_and_poll(
                vector_store_id=VECTOR_STORE_ID,
                file_id=file_id,
            )
        except Exception as e:
            # Falls das File doch schon hängt oder es einen Race gibt, versuchen wir ein retrieve.
            print('Warning: attach failed, trying retrieve:', e)
            vs_file = client.vector_stores.files.retrieve(vector_store_id=VECTOR_STORE_ID, file_id=file_id)

        if getattr(vs_file, 'status', '') == 'failed':
            raise RuntimeError('Vector store indexing failed. Check the OpenAI dashboard / API response details.')
        print('Vector store file status:', getattr(vs_file, 'status', None))
    else:
        try:
            client.vector_stores.files.create(vector_store_id=VECTOR_STORE_ID, file_id=file_id)
        except Exception as e:
            print('Warning: attach failed (continuing):', e)
        print('Attached file to vector store. Waiting for indexing...')
        while True:
            vs = client.vector_stores.retrieve(VECTOR_STORE_ID)
            fc = getattr(vs, 'file_counts', {}) or {}
            if isinstance(fc, dict):
                in_progress = int(fc.get('in_progress', 0) or 0)
                failed = int(fc.get('failed', 0) or 0)
                completed = int(fc.get('completed', 0) or 0)
                total = int(fc.get('total', 0) or 0)
            else:
                in_progress = int(getattr(fc, 'in_progress', 0) or 0)
                failed = int(getattr(fc, 'failed', 0) or 0)
                completed = int(getattr(fc, 'completed', 0) or 0)
                total = int(getattr(fc, 'total', 0) or 0)

            print(f"Indexing status: completed={completed}/{total}, in_progress={in_progress}, failed={failed}")
            if failed:
                raise RuntimeError('Vector store indexing failed. Check the OpenAI dashboard / API response details.')
            if in_progress == 0 and total > 0:
                break
            time.sleep(2)

print('Reuse IDs:')
print('  VECTOR_STORE_ID =', VECTOR_STORE_ID)
print('  PDF_FILE_ID     =', file_id)

print('Vector store ready:', VECTOR_STORE_ID)


In [ ]:
# --- Query: Vector Store durchsuchen + strukturierte Ausgabe (ohne Seitenzahlen) ---

result_schema = {
    'type': 'object',
    'additionalProperties': False,
    'properties': {
        'none_found': {'type': 'boolean'},
        'results': {
            'type': 'array',
            'items': {
                'type': 'object',
                'additionalProperties': False,
                'properties': {
                    'subpoint': {'type': 'string'},
                    'score_1_to_10': {'type': 'integer', 'minimum': 1, 'maximum': 10},
                    'anchor': {'type': 'string'},
                    'summary': {'type': 'string'},
                },
                'required': ['subpoint', 'score_1_to_10', 'anchor', 'summary'],
            },
        },
    },
    'required': ['none_found', 'results'],
}

subpoints_block = ''
if SUBPOINTS:
    subpoints_block = '\n'.join([f"- ({sp.get('id')}) {sp.get('label')}" for sp in SUBPOINTS])
else:
    subpoints_block = '- (Allgemein) Keine Unterpunkte erkannt (du kannst sie in der Beschreibung hinzufügen).'

terms_block = ''
if PREFERRED_SEARCH_TERMS:
    terms_block = ', '.join(PREFERRED_SEARCH_TERMS)

excl_block = ''
if HARD_EXCLUSIONS:
    excl_block = '\n'.join([f"- {x}" for x in HARD_EXCLUSIONS])

search_query = normalize_whitespace(
    f"{CHAPTER_TITLE}\n\n{OPTIMIZED_DESCRIPTION}\n\n"
    f"Keywords: {terms_block}\n\n"
    f"Exclusions:\n{excl_block}"
)

evidence = ''
try:
    search_page = client.vector_stores.search(
        vector_store_id=VECTOR_STORE_ID,
        query=search_query,
        max_num_results=int(FILE_SEARCH_MAX_RESULTS),
        rewrite_query=True,
    )
    evidence = build_evidence_from_vector_store_search(
        search_page,
        max_hits=min(12, int(FILE_SEARCH_MAX_RESULTS)),
        max_chars_per_hit=1800,
    )
except Exception as e:
    print('Warning: evidence retrieval failed:', e)

data = None
if not evidence.strip():
    data = {'none_found': True, 'results': []}
else:
<Prompt entfernt: wird zur Laufzeit aus Firebase geladen>
Du bekommst:
(1) Kapitel-Spezifikation (Titel, Scope, Keywords, Ausschlüsse, Unterpunkte)
(2) EVIDENCE-Auszüge (wörtliche Textstücke aus dem PDF-Index)

WICHTIGSTE REGEL:
- Nutze AUSSCHLIESSLICH Text aus den EVIDENCE-Auszügen.
- Keine externen Fakten, keine Ergänzungen, keine Annahmen.
- Wenn etwas nicht eindeutig in EVIDENCE steht: NICHT aufnehmen.

Ziel:
Finde maximal {MAX_HITS} wirklich passende Stellen für das Kapitel.

Anker-Regel (sehr wichtig):
- 'anchor' MUSS ein wörtliches Zitat aus EVIDENCE sein (8–20 Wörter).
- Exakt kopieren (keine Änderungen, keine Ellipsen, keine Umstellungen).
- Verwende nur normale Leerzeichen (Zeilenumbrüche im Zitat zu Leerzeichen machen).
- Wähle möglichst eine Überschrift oder eine sehr charakteristische Satzphrase, die man per Strg+F findet.

Scoring-Rubrik (1–10):
- 10: Definiert/erklärt den Kernbegriff oder Mechanismus direkt (ideal für Theorieabschnitt).
- 7–9: Substanziell unterstützend (gute Erklärung/Empirie/Begründung im Scope).
- 4–6: Nur teilweise relevant oder sehr kurz/oberflächlich (normalerweise NICHT ausgeben).
- 1–3: Nur erwähnt / Randnotiz (NICHT ausgeben).

Strenge Filter:
- Gib nur Treffer aus, die klar im Scope liegen.
- Alles was primär in den Ausschlüssen liegt: nicht ausgeben.
- Mindestqualität: Gib nur Treffer mit score_1_to_10 >= 7 aus.
- Wenn keine Treffer >= 7 existieren: none_found=true und results=[].

Output:
- Gib NUR JSON zurück, strikt passend zum Schema (keine Markdown-Fences, kein Zusatztext).
"""

    USER2 = f"""### Kapitel
Titel: {CHAPTER_TITLE}

### Such-Spezifikation (optimiert)
{OPTIMIZED_DESCRIPTION}

### Unterpunkte (für Zuordnung)
{subpoints_block}

### Optionale Keywords/Synonyme
{terms_block}

### Ausschlüsse
{excl_block}

### Aufgabe
Analysiere die EVIDENCE-Auszüge und gib nur wirklich passende Stellen zurück.
Für jeden Treffer:
- subpoint: passender Unterpunkt (oder "(Allgemein)", falls keine Unterpunkte sinnvoll sind)
- score_1_to_10: nach Rubrik
- anchor: 8–20 Wörter, exakt aus EVIDENCE kopiert (Strg+F geeignet)
- summary: 2–4 Sätze, nur basierend auf EVIDENCE

### EVIDENCE
{evidence}
"""

    resp = client.responses.create(
        background=False,
        model=model_for_stage('pdf_search', MODEL),
        reasoning={'effort': REASONING_EFFORT},
        input=[
            {'role': 'system', 'content': [{'type': 'input_text', 'text': SYSTEM2}]},
            {'role': 'user', 'content': [{'type': 'input_text', 'text': USER2}]},
        ],
        text={
            'format': {
                'type': 'json_schema',
                'name': 'pdf_findings_evidence',
                'schema': result_schema,
                'strict': True,
            }
        },
        max_output_tokens=int(MAX_OUTPUT_TOKENS),
    )
    resp = poll_response_until_output(client, resp, stage='pdf_search')
    record_cost_event('pdf_search', resp)

    try:
        data = parse_json_from_response(resp, 'pdf_search')
    except Exception as e:
        print('ERROR: could not parse JSON output from model:', e)
        print('--- Raw model output ---')
        print(get_response_text(resp))
        data = {'none_found': True, 'results': []}

def normalize_spaces(text: str) -> str:
    return ' '.join((text or '').split()).strip()

evidence_norm = normalize_spaces(evidence)

raw_results = data.get('results') or []
filtered_results = []
dropped_low_score = 0
dropped_bad_anchor = 0

for r in raw_results:
    try:
        score = int(r.get('score_1_to_10', 0) or 0)
    except Exception:
        score = 0

    if score < 7:
        dropped_low_score += 1
        continue

    anchor_raw = r.get('anchor') or ''
    anchor_norm = normalize_spaces(anchor_raw)
    if not anchor_norm:
        dropped_bad_anchor += 1
        continue
    if anchor_norm != (anchor_raw or '').strip():
        dropped_bad_anchor += 1
        continue

    words = anchor_norm.split(' ')
    if len(words) < 8 or len(words) > 20:
        dropped_bad_anchor += 1
        continue

    if '…' in anchor_norm or '...' in anchor_norm:
        dropped_bad_anchor += 1
        continue

    if evidence_norm and anchor_norm not in evidence_norm:
        dropped_bad_anchor += 1
        continue

    rr = dict(r)
    rr['score_1_to_10'] = int(score)
    rr['anchor'] = anchor_norm
    filtered_results.append(rr)

results = sorted(filtered_results, key=lambda r: int(r.get('score_1_to_10', 0) or 0), reverse=True)
results = results[: int(MAX_HITS)]

if dropped_low_score or dropped_bad_anchor:
    print(f"Note: dropped {dropped_low_score} hits with score<7 and {dropped_bad_anchor} hits with invalid anchor.")

none_found = bool(data.get('none_found')) or not results
if none_found:
    print('KEINE PASSENDEN STELLEN GEFUNDEN')
else:
    for r in results:
        print(f"- Unterpunkt: {r.get('subpoint')}")
        print(f"  - Score (1-10): {r.get('score_1_to_10')}")
        print(f"  - Ort (Suchanker): \"{r.get('anchor')}\"")
        print(f"  - Kurz-Zusammenfassung: {r.get('summary')}")
        print('')


In [ ]:
# --- Cost summary ---
# Wichtig: Das sind Token-Kosten (LLM). Vector-Store Storage/Indexing-Kosten sind NICHT enthalten.

print('--- Pricing (USD / 1M tokens) ---')
for m, p in MODEL_PRICING_USD_PER_1M.items():
    print(f"- {m}: input=${p['input']}/1M, cached_input=${p['cached_input']}/1M, output=${p['output']}/1M")

def pricing_key_for_model_used(model_used: Optional[str]) -> Optional[str]:
    if not model_used:
        return None
    if model_used in MODEL_PRICING_USD_PER_1M:
        return model_used
    # Prefer longest prefix match (handles snapshot names like gpt-5-mini-YYYY-MM-DD)
    matches = [k for k in MODEL_PRICING_USD_PER_1M.keys() if model_used.startswith(k)]
    if not matches:
        return None
    return sorted(matches, key=len, reverse=True)[0]

actual_total = 0.0

print('\n--- Actual run costs (model_used pricing) ---')
for ev in COST_EVENTS:
    stage = ev.get('stage')
    model_used = ev.get('model')
    usage = ev.get('usage') or {}
    costs = ev.get('costs_usd') or {}

    pricing_key = pricing_key_for_model_used(model_used)
    actual_cost = costs.get(pricing_key) if pricing_key else None
    if actual_cost is not None:
        actual_total += float(actual_cost)

    print(f"Stage: {stage} | model_used={model_used} | priced_as={pricing_key or 'n/a'}")
    print(
        f"  tokens: input={usage.get('input_tokens', 0)} "
        f"(cached={usage.get('cached_input_tokens', 0)}), output={usage.get('output_tokens', 0)}"
    )
    print(f"  actual_cost: {fmt_usd(actual_cost)}")
    print('')

print(f"Total actual: ${actual_total:.6f}")

what_if_totals = {m: 0.0 for m in MODEL_PRICING_USD_PER_1M.keys()}
for ev in COST_EVENTS:
    costs = ev.get('costs_usd') or {}
    for m in MODEL_PRICING_USD_PER_1M.keys():
        c = costs.get(m)
        if c is not None:
            what_if_totals[m] += float(c)

print('\n--- What-if totals (same token counts, different pricing) ---')
for m, total in what_if_totals.items():
    print(f"- total@{m}: ${total:.6f}")
